In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

In [2]:
(ds_train, ds_val, ds_test), info = tfds.load(
    "speech_commands",
    split=["train", "validation", "test"],
    as_supervised=True,
    with_info=True
)

commands = info.features["label"].names
print("Total classes:", len(commands))
print(commands)

Total classes: 12
['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes', '_silence_', '_unknown_']


In [3]:
def get_spectrogram(waveform):
    waveform = tf.cast(waveform, tf.float32)
    waveform = tf.reshape(waveform, [-1])

    length = tf.shape(waveform)[0]

    waveform = tf.cond(
        length < 16000,
        lambda: tf.pad(waveform, [[0, 16000 - length]]),
        lambda: waveform[:16000]
    )

    spectrogram = tf.signal.stft(
        waveform,
        frame_length=255,
        frame_step=128
    )

    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.expand_dims(spectrogram, -1)

    return spectrogram


In [4]:
def preprocess(audio, label):
    spec = get_spectrogram(audio)
    return spec, label


In [5]:
batch_size = 64
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    ds_train
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

val_ds = (
    ds_val
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds_test
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
)


In [6]:
for spec, label in train_ds.take(1):
    print("Spectrogram shape:", spec.shape)


Spectrogram shape: (64, 124, 129, 1)


In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(124, 129, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(len(commands), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 122, 127, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 61, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 59, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 29, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 55680)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,127,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,147,532 (27.27 MB)

 Trainable params: 7,147,532 (27.27 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3
1665/1665 ━━━━━━━━━━━━━━━━━━━━ 333s 199ms/step - accuracy: 0.6036 - loss: 134.2149 - val_accuracy: 0.0000e+00 - val_loss: 4.7881
Epoch 2/3
1665/1665 ━━━━━━━━━━━━━━━━━━━━ 337s 202ms/step - accuracy: 0.6317 - loss: 1.5232 - val_accuracy: 0.0000e+00 - val_loss: 4.9610
Epoch 3/3
1665/1665 ━━━━━━━━━━━━━━━━━━━━ 340s 204ms/step - accuracy: 0.6317 - loss: 1.5228 - val_accuracy: 0.0000e+00 - val_loss: 4.9655


In [9]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)


77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step - accuracy: 0.0838 - loss: 3.2127
Test Accuracy: 0.08384457975625992


In [10]:
model.save("speech_commands_model.keras")
print("Model saved")


Model saved


In [15]:
import numpy as np
import soundfile as sf

sr = 16000
t = np.linspace(0, 1, sr, False)
audio = 0.3 * np.sin(2 * np.pi * 440 * t)

sf.write(
    "C:/Users/HP/speech_command_project/sample_audio/yes_real.wav",
    audio,
    sr
)

print("yes_real.wav created")


yes_real.wav created


In [16]:
import librosa


def predict_audio(file_path):
    audio, sr = librosa.load(file_path, sr=16000)

    spec = get_spectrogram(audio)
    spec = tf.expand_dims(spec, 0) 

    prediction = model.predict(spec)
    index = np.argmax(prediction)

    return commands[index], float(np.max(prediction))


In [17]:
label, confidence = predict_audio(
    "C:/Users/HP/speech_command_project/sample_audio/yes_real.wav"
)

print("Predicted:", label)
print("Confidence:", confidence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Predicted: _unknown_
Confidence: 0.8047857880592346
